In [2]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field

In [3]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = openrouter.chat.completions.create(model="openrouter/free", messages=messages)
print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by cohere/north-mini-code:free


Arrr, listen up, ye landlubbers!

The universe be a grand, swirling sea o’ wonder, and its meaning be simple as a bottle o’ rum and salty air:

**To explore, to discover, and to claim yer own treasure!**

Every star be a distant horizon, every planet a hidden cove, every chunk o’ dark matter a ghostly fog awaitin’ yer compass. The cosmos don’t hand ye a ready‑made map; it be up to ye to chart yer course, to set sail on quests o’ science, art, and imagination, and to haul back the spoils—knowledge, wonder, and a good story to spin round the fire.

So hoist yer spyglass, sing a shanty, and remember: the universe ain’t lookin’ for a answer—it’s lookin’ fer a sailor bold enough to keep the adventure goin’! 

Cap’n’s log, clear as a bell: **Find yer own X‑marks, feast on the stars, and share the booty with yer crew.** That’s the meaning o’ it all, ye scallywags! 🌌⚓️🏴‍☠️

In [6]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [7]:
TINY = True

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

5 claims, 5 labels, 50 policies
The policy wording is 1,369 words


In [8]:
claims[0]

{'claim_id': 'CLM-1007',
 'policy_id': 'POL-2007',
 'customer_id': 'CUS-3007',
 'claim_type': 'baggage',
 'submitted_on': '2026-07-22',
 'incident_date': '2026-07-13',
 'incident_location': 'Stockholm, Sweden',
 'summary': 'Phone dropped in the hotel shower',
 'story': 'I dropped my phone in the shower at my hotel in Stockholm on July 13. I know how that sounds. It was on the soap shelf playing music, I was singing along, and during the key change I knocked it off the shelf. It landed screen down in about an inch of water and stayed there longer than it should have, because I had shampoo in my eyes. A repair shop looked at it the next day and said the liquid damage was too extensive to repair. I am claiming the cost of the replacement phone, which was $1,099.',
 'items': [{'description': 'Smartphone, 256 GB, replacement cost',
   'category': 'electronics',
   'amount': 1099.0}],
 'amount_claimed': 1099.0,
 'documents': [{'name': 'Repair shop report',
   'type': 'repair_report',
   'con

In [9]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1007',
 'decision': 'approve',
 'payout': 50.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-07-24',
 'reason': 'Accidental damage during the covered trip, with a repair report stating the phone cannot be repaired. The policy is Economy with no Electronics Plus, so the phone counts at the $150 per-item limit, and the $100 deductible comes off.',
 'reference': 'Policy wording, sections 5 and 10',
 'calculation': [{'step': 'Phone at the per-item limit', 'amount': 150.0},
  {'step': 'Less the deductible', 'amount': -100.0}],
 'tests': 'Payout arithmetic: a per-item limit far below the amount claimed',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'amusing',
 'tiny': True}

In [10]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
1,CLM-1015,Flew home from Paris on arrival after leaving ...,trip_interruption,2340.0,routine,decline,0.0
2,CLM-1017,Backpack snatched by a scooter rider in Naples,baggage,830.0,routine,refer,0.0
3,CLM-1020,Prescription sunglasses stolen by a monkey at ...,baggage,240.0,manual,approve,240.0
4,CLM-1021,Tote bag snatched from her shoulder in Buenos ...,baggage,1103.0,hard,approve,825.0


In [11]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


In [12]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


In [13]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

In [14]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

In [15]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

In [16]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

In [17]:
def decide(claim, model):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "seconds": round(time.time() - start, 1),
    }

In [18]:
first = decide(claims[0], "openrouter/free")
first

{'model': 'openrouter/free',
 'technique': 'LLM',
 'claim_id': 'CLM-1007',
 'decision': 'approve',
 'payout': 50.0,
 'rationale': 'The claim is for accidental damage to a smartphone during the covered trip in Stockholm, which falls under Section 5 (Baggage and Personal Effects). The plan is Economy with no add-ons, so the per-item limit is $150 and the deductible is $100. The repair shop report states the phone is beyond economical repair, satisfying the document requirement for accidental damage. Using Section 10: eligible amount is the lower of $1,099 claimed and the $150 per-item limit = $150. No carrier/other-party payments to subtract. Subtract the $100 deductible: $150 - $100 = $50. This is within the $750 Baggage and Personal Effects maximum. All required documents are attached and the claim was filed within 90 days.',
 'seconds': 62.7}

In [19]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model}: {right} of {len(results)} right in {seconds:.0f} seconds")
    return results

In [20]:
free_results = run("openrouter/free", claims, labels)

openrouter/free: 3 of 5 right in 107 seconds


In [21]:
scored = pd.DataFrame(free_results)
scored.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum")

difficulty,hard,manual,routine
model,,,
openrouter/free,0,0,3


In [22]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(free_results, labels)

**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The plan is First, so Baggage and Personal Effects is covered up to $3,000 with a $500 per-item limit and a $0 deductible. The incident (Aug 19) falls inside the covered trip (Aug 16–28), and the claim was filed within 90 days. The benefit covers loss or theft of personal belongings during the trip.

However, the claim is presented as a theft. Section 5 requires theft to be reported to the local police within 24 hours and a copy of the police report to be provided. No police report was attached; the customer says the temple police post would not take a report about monkeys, and a guide's letter was provided instead. Because a required document is missing and the wording does not clearly settle whether the guide's letter can substitute for the police report (or whether this is better characterized as a non-theft loss), the claim must be referred to a colleague rather than approved or declined outright.

If the colleague decides the evidence is sufficient, Section 10 would give $240 (below the $500 per-item limit, $0 deductible, below the $3,000 maximum). But that payout is not payable on the current documents.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The claim is for theft of baggage during the covered trip, which is covered under section 5 (Baggage and Personal Effects). The police report was provided within 24 hours as required. The items include three electronics with receipts, all under the $1,200 per-item limit due to the Electronics Plus add-on, and the tote bag under the standard $300 per-item limit. The total eligible amount is $1,103. After applying the $50 deductible for the Premium plan, the payout is $1,053. No other deductions apply, and the amount is within the $1,500 maximum for Baggage and Personal Effects.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


In [23]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

User Safety: safe

In [24]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [25]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

In [26]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call_1a35d2a1fc7546bfa6efc354)
 Call ID: call_1a35d2a1fc7546bfa6efc354
  Args:
    policy_id: POL-2007
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2007",
  "customer_id": "CUS-3007",
  "plan": "economy",
  "add_ons": [],
  "purchased_on": "2026-06-02",
  "sold_through": "Northwind Airlines checkout",
  "trip": {
    "destination": "Stockholm, Sweden",
    "departs_on": "2026-07-10",
    "returns_on": "2026-07-17",
    "trip_cost": 2200.0
  },
  "travelers": [
    "Greg Kowalski"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call-bdd663bf-ed7b-499e-8d5c-38d931408b94)
 Call ID: call-bdd663bf-ed7b-499e-8d5c-38d931408b94
  Args:
    decision: decline
    payout: 0
    claim_id: CLM-1007
    rationale: The customer cl

In [27]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    return {
        "model": "openrouter/free",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

5 of 5 claims have a recorded decision


In [28]:
pd.DataFrame(free_results + agent_results).groupby(["model", "technique"], sort=False)[["right", "seconds"]].sum()

right  seconds
model           technique                
openrouter/free LLM            3    106.7
                Agent          3    175.2

In [29]:
show_misses(agent_results, labels)

**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The theft of prescription sunglasses during a covered trip in Bali falls under Section 5 (Baggage and Personal Effects), and the $240 claim is within the $500 per-item limit for the First plan. However, Section 5 requires theft to be reported to local police within 24 hours and a copy of the police report to be provided. The customer attempted to report at the temple gate police post but was refused because the thieves were monkeys, and a guide's letter was provided instead. A required document (police report) is missing, and the exceptional circumstances warrant colleague review to determine whether the policy requirement can be waived.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The policy covers theft of personal belongings during the covered trip under Section 5. All required conditions are met: theft reported to local police within 24 hours with police report attached, items are not excluded (not unattended, cash below $100, etc.), and claim is filed within 90 days. Premium plan allows $1,500 maximum for Baggage and Personal Effects with $300 per-item limit, and Electronics Plus add-on applies (though not needed here as all items are under limit). Payout calculated as claimed amount ($1,103) minus deductible ($50) = $1,053, which is under the $1,500 maximum.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


In [30]:
rows = []
for results in [free_results, agent_results]:
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,technique,accuracy,speed
0,openrouter/free,LLM,3 / 5,21.3 seconds a claim
1,openrouter/free,Agent,3 / 5,35.0 seconds a claim
